# 第5回: RNN を用いた時系列予測学習

このノートブックでは、リカレントニューラルネットワーク（RNN）を用いた時系列予測について学びます。
ロボットの動作学習では、時系列データ（関節角度や座標の変化）を扱うことが不可欠です。
RNNは過去の情報を「隠れ状態」として保持し、時系列パターンを学習できるモデルです。

**学習内容:**
1. **RNN・LSTMの基本概念** — 隠れ状態の伝播とゲート機構
2. **BPTT（時間方向の誤差逆伝播）** — RNNの学習アルゴリズム
3. **Robomimicデータでの学習と可視化** — 実データへの適用
4. **番外編: MTRNN** — 多重時定数RNNの実装

## 環境準備

必要なライブラリをインポートします。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False
import torch
from tqdm.notebook import tqdm
import torch.nn as nn
import torch.optim as optim
import os

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

---
## 1. RNN・LSTMの基本概念

### 1.1 RNNとは？

RNN（Recurrent Neural Network）は、時系列データを扱うためのニューラルネットワークです。
通常のフィードフォワードネットワークと異なり、**隠れ状態** $h_t$ を通じて過去の情報を次の時刻に伝えることができます。

$$
h_t = f(W_{ih} x_t + W_{hh} h_{t-1} + b)
$$

ここで：
- $x_t$ : 時刻 $t$ の入力
- $h_{t-1}$ : 前の時刻の隠れ状態
- $W_{ih}, W_{hh}$ : 重み行列
- $f$ : 活性化関数（通常は tanh）

### 隠れ状態の伝播（展開図）

```
時刻:    t=0       t=1       t=2       t=3
入力:    x_0       x_1       x_2       x_3
          │         │         │         │
          ▼         ▼         ▼         ▼
隠れ: h_0 → [RNN] → h_1 → [RNN] → h_2 → [RNN] → h_3 → [RNN] → h_4
                      │         │         │         │
                      ▼         ▼         ▼         ▼
出力:               y_1       y_2       y_3       y_4
```

各時刻のRNNユニットは**同じ重みを共有**しています。
これにより、任意の長さの系列を同じモデルで処理できます。

### 1.2 RNNの問題点：勾配消失

通常のRNNには**勾配消失問題**があります。
長い時系列を学習する際、誤差が時間方向に逆伝播する過程で勾配が指数的に小さくなり、
初期の時刻の情報が学習に反映されなくなります。

$$
\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}} \approx 0 \quad (T \text{が大きい場合})
$$

### 1.3 LSTM（Long Short-Term Memory）

LSTMは勾配消失問題を解決するために設計されたRNNの一種です。
**セル状態** $c_t$ という「高速道路」を追加し、3つの**ゲート**で情報の流れを制御します。

| ゲート | 役割 | 数式 |
|--------|------|------|
| **忘却ゲート** $f_t$ | 古い情報をどれだけ忘れるか | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ |
| **入力ゲート** $i_t$ | 新しい情報をどれだけ記憶するか | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ |
| **出力ゲート** $o_t$ | セル状態のどの部分を出力するか | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ |

セル状態と隠れ状態の更新：
$$
\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)
$$
$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$
$$
h_t = o_t \odot \tanh(c_t)
$$

$\sigma$ はシグモイド関数、$\odot$ は要素積です。
セル状態 $c_t$ は加算的に更新されるため、勾配が消失しにくくなります。

### 1.4 RNNCell vs LSTMCell の比較

PyTorchでは `nn.RNNCell` と `nn.LSTMCell` が提供されています。

| 特徴 | `nn.RNNCell` | `nn.LSTMCell` |
|------|-------------|---------------|
| 隠れ状態 | $h_t$ のみ | $(h_t, c_t)$ のタプル |
| ゲート機構 | なし | 忘却・入力・出力ゲート |
| 長期記憶 | 苦手（勾配消失） | 得意（セル状態による保持） |
| パラメータ数 | 少ない | 多い（約4倍） |
| 用途 | 短い系列 | 長い系列、ロボット動作学習 |

ロボットの動作学習では、数百ステップ以上の系列を扱うことが多いため、**LSTMが標準的に使用**されます。

### 1.5 BasicLSTM モデルの実装

LSTMCellを使ったシンプルなモデルを実装します。

In [ ]:
class BasicLSTM(nn.Module):
    def __init__(self, in_dim, rec_dim, out_dim):
        """LSTM を用いた時系列予測モデル

        Args:
            in_dim: 入力次元（例: 関節角度の数）
            rec_dim: LSTMの隠れ層次元
            out_dim: 出力次元（予測する値の数）
        """
        super().__init__()
        self.rnn = nn.LSTMCell(in_dim, rec_dim)
        self.output = nn.Sequential(
            nn.Linear(rec_dim, out_dim),
            nn.Tanh()
        )

    def forward(self, x, state=None):
        """1タイムステップの前向き計算

        Args:
            x: 入力テンソル (batch_size, in_dim)
            state: LSTMの状態 (h, c) のタプル、Noneなら初期化

        Returns:
            y_hat: 予測出力 (batch_size, out_dim)
            rnn_hid: 更新されたLSTM状態 (h, c)
        """
        rnn_hid = self.rnn(x, state)
        y_hat = self.output(rnn_hid[0])  # h (隠れ状態) を出力層に渡す
        return y_hat, rnn_hid

### モデルの構造確認

In [ ]:
# モデルの作成と構造確認
in_dim = 7    # 例: 7自由度ロボットアームの関節角度
rec_dim = 50  # LSTM隠れ層の次元
out_dim = 7   # 出力も同じ7次元

model = BasicLSTM(in_dim, rec_dim, out_dim)
print(model)
print(f"\n総パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

# 1ステップの動作確認
x_sample = torch.randn(1, in_dim)  # バッチサイズ1、入力次元7
y_hat, state = model(x_sample)
print(f"\n入力形状: {x_sample.shape}")
print(f"出力形状: {y_hat.shape}")
print(f"隠れ状態 h の形状: {state[0].shape}")
print(f"セル状態 c の形状: {state[1].shape}")

---
## 2. BPTT（Backpropagation Through Time）

### 2.1 BPTTの概念

BPTT（時間方向の誤差逆伝播）は、RNNを学習するための標準的なアルゴリズムです。
時系列全体にわたって計算グラフを展開し、各時刻の損失から勾配を計算します。

```
順伝播（Forward）：
  x_0 → [RNN] → y_0 → Loss_0
           ↓ h_0
  x_1 → [RNN] → y_1 → Loss_1
           ↓ h_1
  x_2 → [RNN] → y_2 → Loss_2
           ↓ h_2
  ...
  x_T → [RNN] → y_T → Loss_T

         Total Loss = (1/T) Σ Loss_t

逆伝播（Backward）：
  Total Loss から時間を遡って各パラメータの勾配を計算
  ←── ←── ←── ←── ←── (時間方向に逆伝播)
```

### 2.2 Full BPTT

Full BPTTでは、系列全体にわたって勾配を計算します。
- **利点**: 長期依存関係を正確に学習できる
- **欠点**: メモリ使用量が系列長に比例して増加する

ロボットの動作系列（100〜500ステップ程度）では、Full BPTTが一般的に使用されます。

### 2.3 fullBPTTtrainer の実装

Full BPTTによる学習を行うトレーナークラスを実装します。

In [ ]:
class fullBPTTtrainer:
    """Full BPTT による RNN 学習トレーナー

    時系列全体を順伝播した後、累積損失から一括で逆伝播を行う。
    """
    def __init__(self, model, optimizer, device="cpu"):
        self.device = device
        self.optimizer = optimizer
        self.model = model.to(device)

    def process_epoch(self, x_data, y_data):
        """1エポックの学習を実行

        Args:
            x_data: 入力系列 (batch_size, T, in_dim)
            y_data: 目標系列 (batch_size, T, out_dim)

        Returns:
            float: 平均損失値
        """
        self.model.train()
        total_loss = 0.0
        self.optimizer.zero_grad()

        state = None
        T = x_data.shape[1]  # 系列長

        # 時刻 t=0 から T-1 まで順伝播
        for t in range(T):
            y_hat, state = self.model(x_data[:, t], state)
            total_loss += nn.MSELoss()(y_hat, y_data[:, t])

        # 平均損失
        total_loss /= T

        # 時間方向に一括で逆伝播（Full BPTT）
        total_loss.backward()

        # パラメータ更新
        self.optimizer.step()

        return total_loss.item()

### 2.4 ダミーデータでの動作確認

まず、簡単な正弦波データで学習が動作することを確認します。

In [ ]:
# ダミーデータの作成（正弦波の予測）
def create_sine_data(n_samples=10, seq_len=100, n_dim=3, noise=0.01):
    """正弦波の時系列データを作成"""
    t = np.linspace(0, 4 * np.pi, seq_len + 1)
    data = np.zeros((n_samples, seq_len + 1, n_dim))
    for i in range(n_samples):
        for d in range(n_dim):
            freq = 0.5 + np.random.rand() * 1.5  # ランダムな周波数
            phase = np.random.rand() * 2 * np.pi  # ランダムな位相
            data[i, :, d] = np.sin(freq * t + phase)
    data += np.random.randn(*data.shape) * noise
    # Tanhの出力範囲に合わせて [-0.9, 0.9] にクリップ
    data = np.clip(data, -0.9, 0.9)
    return data

sine_data = create_sine_data(n_samples=10, seq_len=100, n_dim=3)
x_sine = torch.tensor(sine_data[:, :-1], dtype=torch.float32)  # t=0..T-1
y_sine = torch.tensor(sine_data[:, 1:], dtype=torch.float32)   # t=1..T

print(f"入力データ形状: {x_sine.shape}  (バッチ, 系列長, 次元)")
print(f"目標データ形状: {y_sine.shape}")

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for d in range(3):
    axes[d].plot(sine_data[0, :, d], label=f"次元 {d}")
    axes[d].set_xlabel("時刻 t")
    axes[d].set_ylabel("値")
    axes[d].set_title(f"サンプル0 - 次元{d}")
    axes[d].grid(True, alpha=0.3)
plt.suptitle("ダミーデータ（正弦波）", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ダミーデータでの学習テスト
test_model = BasicLSTM(in_dim=3, rec_dim=30, out_dim=3)
test_optimizer = optim.Adam(test_model.parameters(), lr=0.001)
trainer = fullBPTTtrainer(test_model, test_optimizer)

losses = []
for epoch in tqdm(range(50), desc="テスト学習"):
    loss = trainer.process_epoch(x_sine, y_sine)
    losses.append(loss)

plt.plot(losses)
plt.xlabel("エポック")
plt.ylabel("損失 (MSE)")
plt.title("ダミーデータでの学習曲線")
plt.grid(True, alpha=0.3)
plt.show()
print("学習が正常に動作することを確認しました。")

---
## 3. Robomimicデータを使った学習と可視化

### 3.1 データの読み込み

第4回で作成したLeRobot形式のNumPyデータセットを読み込みます。
データが存在しない場合は、ダミーデータを作成して学習を進めます。

In [ ]:
# LeRobot形式データの読み込み
data_dir = "data"

if os.path.exists(os.path.join(data_dir, "observation.state.npy")) and \
   os.path.exists(os.path.join(data_dir, "action.npy")):
    print("LeRobot形式データを読み込みます...")
    states = np.load(os.path.join(data_dir, "observation.state.npy"))   # (N, T, state_dim)
    actions = np.load(os.path.join(data_dir, "action.npy"))             # (N, T, action_dim)
    print(f"状態データ形状: {states.shape}")
    print(f"行動データ形状: {actions.shape}")
else:
    print("データが見つかりません。ダミーデータを作成します。")
    print("（第4回のノートブックでデータを作成してください）")
    N, T, state_dim = 20, 100, 7
    # ダミーデータ: 複数の正弦波を組み合わせたロボット軌道
    t_axis = np.linspace(0, 2 * np.pi, T)
    states = np.zeros((N, T, state_dim))
    for i in range(N):
        for d in range(state_dim):
            amp = 0.3 + 0.5 * np.random.rand()
            freq = 0.8 + 1.5 * np.random.rand()
            phase = np.random.rand() * 2 * np.pi
            states[i, :, d] = amp * np.sin(freq * t_axis + phase)
    states += np.random.randn(*states.shape) * 0.01
    states = np.clip(states, -0.9, 0.9).astype(np.float32)
    actions = np.diff(states, axis=1)  # 行動 = 状態の差分
    actions = np.concatenate([actions, actions[:, -1:]], axis=1).astype(np.float32)
    print(f"ダミー状態データ形状: {states.shape}")
    print(f"ダミー行動データ形状: {actions.shape}")

### 3.2 データの前処理

状態データを入力（現在の状態）と目標（次の状態）に分割します。
RNNは現在の状態を入力として受け取り、次の時刻の状態を予測するように学習します。

```
入力: x = states[:, 0:T-1]   （時刻 0 から T-2）
目標: y = states[:, 1:T]     （時刻 1 から T-1）
```

In [ ]:
# 状態データから入出力ペアを作成
x_data = torch.tensor(states[:, :-1], dtype=torch.float32)  # 現在の状態
y_data = torch.tensor(states[:, 1:], dtype=torch.float32)   # 次の状態

# 訓練データとテストデータに分割
n_train = int(len(x_data) * 0.8)
x_train, x_test = x_data[:n_train], x_data[n_train:]
y_train, y_test = y_data[:n_train], y_data[n_train:]

print(f"訓練データ: x={x_train.shape}, y={y_train.shape}")
print(f"テストデータ: x={x_test.shape}, y={y_test.shape}")
print(f"入力次元: {x_train.shape[-1]}")
print(f"系列長: {x_train.shape[1]}")

### 3.3 モデルの学習

200エポックの学習を実行し、損失の変化を記録します。

In [ ]:
# モデルの設定
state_dim = x_train.shape[-1]
rec_dim = 50      # LSTM隠れ層の次元
out_dim = state_dim

model = BasicLSTM(state_dim, rec_dim, out_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer = fullBPTTtrainer(model, optimizer, device="cpu")

# 学習ループ
n_epochs = 200
train_losses = []
test_losses = []

pbar = tqdm(range(n_epochs), desc="LSTM学習")
for epoch in pbar:
    # 訓練
    train_loss = trainer.process_epoch(x_train, y_train)
    train_losses.append(train_loss)

    # テスト（勾配計算なし）
    model.eval()
    with torch.no_grad():
        test_loss = 0.0
        state = None
        T = x_test.shape[1]
        for t in range(T):
            y_hat, state = model(x_test[:, t], state)
            test_loss += nn.MSELoss()(y_hat, y_test[:, t]).item()
        test_loss /= T
    test_losses.append(test_loss)

    pbar.set_postfix(train=f"{train_loss:.6f}", test=f"{test_loss:.6f}")

print("学習完了！")

### 3.4 損失曲線の可視化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 全体の損失曲線
axes[0].plot(train_losses, label="訓練損失", alpha=0.8)
axes[0].plot(test_losses, label="テスト損失", alpha=0.8)
axes[0].set_xlabel("エポック")
axes[0].set_ylabel("損失 (MSE)")
axes[0].set_title("学習曲線")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 対数スケール
axes[1].semilogy(train_losses, label="訓練損失", alpha=0.8)
axes[1].semilogy(test_losses, label="テスト損失", alpha=0.8)
axes[1].set_xlabel("エポック")
axes[1].set_ylabel("損失 (MSE, 対数)")
axes[1].set_title("学習曲線（対数スケール）")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"最終訓練損失: {train_losses[-1]:.6f}")
print(f"最終テスト損失: {test_losses[-1]:.6f}")

### 3.5 予測と実測の比較

テストデータに対するモデルの予測と、実際の軌道を各関節ごとに可視化します。

In [ ]:
# テストデータでの予測
model.eval()
sample_idx = 0  # 可視化するサンプルのインデックス

predictions = []
state = None
with torch.no_grad():
    for t in range(x_test.shape[1]):
        y_hat, state = model(x_test[sample_idx:sample_idx+1, t], state)
        predictions.append(y_hat.numpy())

predictions = np.concatenate(predictions, axis=0)  # (T, out_dim)
ground_truth = y_test[sample_idx].numpy()            # (T, out_dim)

print(f"予測データ形状: {predictions.shape}")
print(f"正解データ形状: {ground_truth.shape}")

In [ ]:
# 各関節の予測と実測の比較
n_joints = ground_truth.shape[1]
n_cols = min(n_joints, 4)
n_rows = (n_joints + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1) if n_cols > 1 else np.array([[axes]])
if n_cols == 1:
    axes = axes.reshape(-1, 1)

time_steps = np.arange(ground_truth.shape[0])

for d in range(n_joints):
    r, c = d // n_cols, d % n_cols
    ax = axes[r, c]
    ax.plot(time_steps, ground_truth[:, d], label="実測", alpha=0.8, linewidth=2)
    ax.plot(time_steps, predictions[:, d], label="予測", alpha=0.8, linewidth=2, linestyle="--")
    ax.set_xlabel("時刻 t")
    ax.set_ylabel("値")
    ax.set_title(f"関節 {d}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# 余ったサブプロットを非表示
for d in range(n_joints, n_rows * n_cols):
    r, c = d // n_cols, d % n_cols
    axes[r, c].set_visible(False)

plt.suptitle("予測 vs 実測（テストデータ）", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 各関節の予測誤差
errors = np.abs(predictions - ground_truth)
print("各関節の平均絶対誤差 (MAE):")
for d in range(n_joints):
    print(f"  関節 {d}: {errors[:, d].mean():.6f}")
print(f"  全体平均: {errors.mean():.6f}")

---
## 4. 番外編: MTRNN（Multiple Timescale RNN）

### 4.1 MTRNNの概念

MTRNN（多重時定数RNN）は、異なる時間スケールで情報を処理するRNNです。
通常のRNNやLSTMは単一の時間スケールで動作しますが、
MTRNNは**Fast（速い）コンテキスト**と**Slow（遅い）コンテキスト**の2層を持ちます。

```
入力 x_t
  │
  ▼
[Fast Context] ←→ [Slow Context]
  (tau_fast)        (tau_slow)
  │
  ▼
出力 y_t
```

### 4.2 時定数（tau）の役割

各層には**時定数 $\tau$** が設定されます。時定数は情報の更新速度を制御します：

$$
u_t = \left(1 - \frac{1}{\tau}\right) u_{t-1} + \frac{1}{\tau} \tilde{u}_t
$$
$$
h_t = \tanh(u_t)
$$

- **$\tau$ が小さい**（例: $\tau = 2$）: 高速に変化 → 細かい動きのパターン
- **$\tau$ が大きい**（例: $\tau = 20$）: ゆっくり変化 → 動作全体の構造

### 4.3 ロボット学習における利点

| 層 | 時定数 | 担当 |
|----|--------|------|
| Fast Context | 小 ($\tau \approx 2$) | 関節の瞬間的な動き |
| Slow Context | 大 ($\tau \approx 20$) | 動作の種類、全体的な軌道パターン |

例えば、「物を掴む」「物を置く」といった異なる動作パターンを、
Slow Contextが動作の種類を表現し、Fast Contextが具体的な軌道を生成する、
という階層的な表現が自然に獲得されます。

### 4.4 MTRNNCellの実装

FastとSlowの2層を持つMTRNNセルを実装します。

In [ ]:
class MTRNNCell(nn.Module):
    """多重時定数RNN (MTRNN) セル

    Fast/Slow の2つのコンテキスト層を持ち、
    それぞれ異なる時定数で情報を更新する。
    """
    def __init__(self, input_dim, fast_dim, slow_dim, fast_tau, slow_tau):
        super().__init__()
        self.fast_dim = fast_dim
        self.slow_dim = slow_dim
        self.fast_tau = fast_tau
        self.slow_tau = slow_tau

        # 入力 → Fast
        self.i2f = nn.Linear(input_dim, fast_dim)
        # Fast → Fast（自己結合）
        self.f2f = nn.Linear(fast_dim, fast_dim, bias=False)
        # Fast → Slow
        self.f2s = nn.Linear(fast_dim, slow_dim)
        # Slow → Slow（自己結合）
        self.s2s = nn.Linear(slow_dim, slow_dim, bias=False)
        # Slow → Fast
        self.s2f = nn.Linear(slow_dim, fast_dim)

    def forward(self, x, state=None):
        """1タイムステップの前向き計算

        Args:
            x: 入力 (batch_size, input_dim)
            state: (h_fast, h_slow, u_fast, u_slow) のタプル

        Returns:
            (new_h_fast, new_h_slow, new_u_fast, new_u_slow)
        """
        batch_size = x.shape[0]

        if state is not None:
            prev_h_fast, prev_h_slow, prev_u_fast, prev_u_slow = state
        else:
            device = x.device
            prev_h_fast = torch.zeros(batch_size, self.fast_dim, device=device)
            prev_h_slow = torch.zeros(batch_size, self.slow_dim, device=device)
            prev_u_fast = torch.zeros(batch_size, self.fast_dim, device=device)
            prev_u_slow = torch.zeros(batch_size, self.slow_dim, device=device)

        # Fast context の更新
        new_u_fast = (1 - 1/self.fast_tau) * prev_u_fast + \
                     (1/self.fast_tau) * (self.i2f(x) + self.f2f(prev_h_fast) + self.s2f(prev_h_slow))

        # Slow context の更新
        new_u_slow = (1 - 1/self.slow_tau) * prev_u_slow + \
                     (1/self.slow_tau) * (self.f2s(prev_h_fast) + self.s2s(prev_h_slow))

        # 活性化
        new_h_fast = torch.tanh(new_u_fast)
        new_h_slow = torch.tanh(new_u_slow)

        return new_h_fast, new_h_slow, new_u_fast, new_u_slow

### 4.5 BasicMTRNNモデル

In [ ]:
class BasicMTRNN(nn.Module):
    """MTRNN を用いた時系列予測モデル"""
    def __init__(self, in_dim, fast_dim, slow_dim, fast_tau, slow_tau, out_dim):
        super().__init__()
        self.mtrnn = MTRNNCell(in_dim, fast_dim, slow_dim, fast_tau, slow_tau)
        self.output = nn.Sequential(
            nn.Linear(fast_dim, out_dim),
            nn.Tanh()
        )

    def forward(self, x, state=None):
        """1タイムステップの前向き計算

        Args:
            x: 入力 (batch_size, in_dim)
            state: MTRNNの状態タプル

        Returns:
            y_hat: 予測出力
            rnn_hid: 更新された状態
        """
        rnn_hid = self.mtrnn(x, state)
        y_hat = self.output(rnn_hid[0])  # Fast context から出力
        return y_hat, rnn_hid

### 4.6 モデルの切り替え

引数によってLSTMとMTRNNを簡単に切り替えることができます。

In [ ]:
# argparse の代わりに辞書で引数を管理（ノートブック用）
class Args:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

# ---- LSTM の場合 ----
args_lstm = Args(
    model_type="LSTM",
    in_dim=state_dim,
    rec_dim=50,
    out_dim=state_dim
)

# ---- MTRNN の場合 ----
args_mtrnn = Args(
    model_type="MTRNN",
    in_dim=state_dim,
    fast_dim=40,
    slow_dim=10,
    fast_tau=2,
    slow_tau=20,
    out_dim=state_dim
)

def create_model(args):
    """args引数でLSTMとMTRNNを切り替え"""
    if args.model_type == "LSTM":
        model = BasicLSTM(args.in_dim, args.rec_dim, args.out_dim)
    elif args.model_type == "MTRNN":
        model = BasicMTRNN(args.in_dim, args.fast_dim, args.slow_dim,
                           args.fast_tau, args.slow_tau, args.out_dim)
    else:
        raise ValueError(f"未知のモデルタイプ: {args.model_type}")
    return model

# 動作確認
for args in [args_lstm, args_mtrnn]:
    m = create_model(args)
    x_test_sample = torch.randn(1, state_dim)
    y_out, s = m(x_test_sample)
    print(f"{args.model_type:6s}: 出力形状={y_out.shape}, "
          f"パラメータ数={sum(p.numel() for p in m.parameters()):,}")

### 4.7 MTRNNの学習とLSTMとの比較

In [ ]:
# MTRNN モデルの学習
mtrnn_model = create_model(args_mtrnn)
mtrnn_optimizer = optim.Adam(mtrnn_model.parameters(), lr=0.001)
mtrnn_trainer = fullBPTTtrainer(mtrnn_model, mtrnn_optimizer, device="cpu")

mtrnn_losses = []
for epoch in tqdm(range(n_epochs), desc="MTRNN学習"):
    loss = mtrnn_trainer.process_epoch(x_train, y_train)
    mtrnn_losses.append(loss)

print("学習完了！")

# 損失曲線の比較
plt.figure(figsize=(10, 4))
plt.semilogy(train_losses, label="LSTM (訓練)", alpha=0.8)
plt.semilogy(mtrnn_losses, label="MTRNN (訓練)", alpha=0.8)
plt.xlabel("エポック")
plt.ylabel("損失 (MSE, 対数)")
plt.title("LSTM vs MTRNN 学習曲線の比較")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"LSTM  最終損失: {train_losses[-1]:.6f}")
print(f"MTRNN 最終損失: {mtrnn_losses[-1]:.6f}")

### 4.8 Fast/Slow コンテキストの可視化

MTRNNの特徴として、Fast Contextの活性が高速に変化し、
Slow Contextの活性がゆっくり変化することが確認できます。

In [ ]:
# MTRNNの内部状態を可視化
mtrnn_model.eval()
fast_activations = []
slow_activations = []

state = None
with torch.no_grad():
    for t in range(x_test.shape[1]):
        y_hat, state = mtrnn_model(x_test[0:1, t], state)
        fast_activations.append(state[0].numpy().copy())
        slow_activations.append(state[1].numpy().copy())

fast_activations = np.concatenate(fast_activations, axis=0)  # (T, fast_dim)
slow_activations = np.concatenate(slow_activations, axis=0)  # (T, slow_dim)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Fast Context（最初の5ニューロン）
n_show = min(5, fast_activations.shape[1])
for i in range(n_show):
    axes[0].plot(fast_activations[:, i], alpha=0.7, label=f"Fast #{i}")
axes[0].set_xlabel("時刻 t")
axes[0].set_ylabel("活性値")
axes[0].set_title(f"Fast Context (tau={args_mtrnn.fast_tau}) - 高速に変動")
axes[0].legend(fontsize=8, ncol=5)
axes[0].grid(True, alpha=0.3)

# Slow Context（最初の5ニューロン）
n_show = min(5, slow_activations.shape[1])
for i in range(n_show):
    axes[1].plot(slow_activations[:, i], alpha=0.7, label=f"Slow #{i}")
axes[1].set_xlabel("時刻 t")
axes[1].set_ylabel("活性値")
axes[1].set_title(f"Slow Context (tau={args_mtrnn.slow_tau}) - ゆっくり変動")
axes[1].legend(fontsize=8, ncol=5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 演習問題

以下の演習で、今回学んだRNNの概念を実践的に理解しましょう。

### 演習1: BasicLSTMモデルのforward関数を実装せよ

以下の `MyBasicLSTM` クラスの `forward` メソッドを実装してください。
- `self.rnn` (LSTMCell) に入力 `x` と状態 `state` を渡す
- LSTMCellの出力は `(h, c)` のタプルであることに注意
- 隠れ状態 `h` を `self.output` 層に通して予測値 `y_hat` を得る
- `y_hat` とLSTMの状態を返す

In [ ]:
class MyBasicLSTM(nn.Module):
    def __init__(self, in_dim, rec_dim, out_dim):
        super().__init__()
        self.rnn = nn.LSTMCell(in_dim, rec_dim)
        self.output = nn.Sequential(
            nn.Linear(rec_dim, out_dim),
            nn.Tanh()
        )

    def forward(self, x, state=None):
        # ここにコードを書いてください
        pass

# テスト
my_model = MyBasicLSTM(in_dim=7, rec_dim=50, out_dim=7)
x_test_input = torch.randn(1, 7)
y_out, new_state = my_model(x_test_input)
print(f"出力形状: {y_out.shape}")
print(f"隠れ状態形状: {new_state[0].shape}")
print(f"セル状態形状: {new_state[1].shape}")

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyBasicLSTM(nn.Module):
    def __init__(self, in_dim, rec_dim, out_dim):
        super().__init__()
        self.rnn = nn.LSTMCell(in_dim, rec_dim)
        self.output = nn.Sequential(
            nn.Linear(rec_dim, out_dim),
            nn.Tanh()
        )

    def forward(self, x, state=None):
        rnn_hid = self.rnn(x, state)       # (h, c) のタプル
        y_hat = self.output(rnn_hid[0])     # h を出力層に通す
        return y_hat, rnn_hid

# テスト
my_model = MyBasicLSTM(in_dim=7, rec_dim=50, out_dim=7)
x_test_input = torch.randn(1, 7)
y_out, new_state = my_model(x_test_input)
print(f"出力形状: {y_out.shape}")
print(f"隠れ状態形状: {new_state[0].shape}")
print(f"セル状態形状: {new_state[1].shape}")
```

</details>

### 演習2: fullBPTTtrainerのprocess_epochを実装し、ダミーデータで学習せよ

以下の `MyTrainer` クラスの `process_epoch` メソッドを実装してください。
- `self.optimizer.zero_grad()` で勾配をリセット
- 各時刻 `t` で `self.model` に入力を渡し、MSE損失を累積
- 系列長 `T` で割って平均損失を計算
- `backward()` と `optimizer.step()` で学習
- 損失値 (float) を返す

実装後、作成済みの正弦波データ (`x_sine`, `y_sine`) で100エポック学習し、損失曲線をプロットしてください。

In [ ]:
class MyTrainer:
    def __init__(self, model, optimizer, device="cpu"):
        self.device = device
        self.optimizer = optimizer
        self.model = model.to(device)

    def process_epoch(self, x_data, y_data):
        self.model.train()
        # ここにコードを書いてください
        pass

# テスト: 正弦波データで学習
my_model = BasicLSTM(in_dim=3, rec_dim=30, out_dim=3)
my_opt = optim.Adam(my_model.parameters(), lr=0.001)
my_trainer = MyTrainer(my_model, my_opt)

# ここにコードを書いてください（100エポック学習と損失曲線のプロット）

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyTrainer:
    def __init__(self, model, optimizer, device="cpu"):
        self.device = device
        self.optimizer = optimizer
        self.model = model.to(device)

    def process_epoch(self, x_data, y_data):
        self.model.train()
        total_loss = 0.0
        self.optimizer.zero_grad()
        state = None
        T = x_data.shape[1]
        for t in range(T):
            y_hat, state = self.model(x_data[:, t], state)
            total_loss += nn.MSELoss()(y_hat, y_data[:, t])
        total_loss /= T
        total_loss.backward()
        self.optimizer.step()
        return total_loss.item()

# 学習の実行
my_model = BasicLSTM(in_dim=3, rec_dim=30, out_dim=3)
my_opt = optim.Adam(my_model.parameters(), lr=0.001)
my_trainer = MyTrainer(my_model, my_opt)

losses = []
for epoch in range(100):
    loss = my_trainer.process_epoch(x_sine, y_sine)
    losses.append(loss)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.6f}")

plt.plot(losses)
plt.xlabel("エポック")
plt.ylabel("損失 (MSE)")
plt.title("ダミーデータでの学習曲線")
plt.grid(True, alpha=0.3)
plt.show()
```

</details>

### 演習3: （番外編）MTRNNを実装し、LSTMとの予測精度を比較せよ

`MTRNNCell` と `BasicMTRNN` を参考に、自分でMTRNNモデルを実装してください。
以下の手順で進めてください：

1. `MTRNNCell` を実装する（Fast: dim=40, tau=2、Slow: dim=10, tau=20）
2. `BasicMTRNN` を実装する
3. `fullBPTTtrainer` を使って200エポック学習する
4. LSTMの損失曲線と重ねてプロットし、精度を比較する

**ヒント:**
- `MTRNNCell` は `(h_fast, h_slow, u_fast, u_slow)` の4つの状態を管理する
- 時定数の更新式: $u_t = (1 - 1/\tau) u_{t-1} + (1/\tau) \tilde{u}_t$
- `BasicMTRNN` の出力は Fast Context の `h_fast` から生成する

In [ ]:
class MyMTRNNCell(nn.Module):
    def __init__(self, input_dim, fast_dim, slow_dim, fast_tau, slow_tau):
        super().__init__()
        # ここにコードを書いてください
        pass

    def forward(self, x, state=None):
        # ここにコードを書いてください
        pass

class MyBasicMTRNN(nn.Module):
    def __init__(self, in_dim, fast_dim, slow_dim, fast_tau, slow_tau, out_dim):
        super().__init__()
        # ここにコードを書いてください
        pass

    def forward(self, x, state=None):
        # ここにコードを書いてください
        pass

# テスト: 学習と比較
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyMTRNNCell(nn.Module):
    def __init__(self, input_dim, fast_dim, slow_dim, fast_tau, slow_tau):
        super().__init__()
        self.fast_dim = fast_dim
        self.slow_dim = slow_dim
        self.fast_tau = fast_tau
        self.slow_tau = slow_tau
        self.i2f = nn.Linear(input_dim, fast_dim)
        self.f2f = nn.Linear(fast_dim, fast_dim, bias=False)
        self.f2s = nn.Linear(fast_dim, slow_dim)
        self.s2s = nn.Linear(slow_dim, slow_dim, bias=False)
        self.s2f = nn.Linear(slow_dim, fast_dim)

    def forward(self, x, state=None):
        batch_size = x.shape[0]
        if state is not None:
            prev_h_fast, prev_h_slow, prev_u_fast, prev_u_slow = state
        else:
            device = x.device
            prev_h_fast = torch.zeros(batch_size, self.fast_dim, device=device)
            prev_h_slow = torch.zeros(batch_size, self.slow_dim, device=device)
            prev_u_fast = torch.zeros(batch_size, self.fast_dim, device=device)
            prev_u_slow = torch.zeros(batch_size, self.slow_dim, device=device)
        new_u_fast = (1 - 1/self.fast_tau) * prev_u_fast + \
                     (1/self.fast_tau) * (self.i2f(x) + self.f2f(prev_h_fast) + self.s2f(prev_h_slow))
        new_u_slow = (1 - 1/self.slow_tau) * prev_u_slow + \
                     (1/self.slow_tau) * (self.f2s(prev_h_fast) + self.s2s(prev_h_slow))
        new_h_fast = torch.tanh(new_u_fast)
        new_h_slow = torch.tanh(new_u_slow)
        return new_h_fast, new_h_slow, new_u_fast, new_u_slow

class MyBasicMTRNN(nn.Module):
    def __init__(self, in_dim, fast_dim, slow_dim, fast_tau, slow_tau, out_dim):
        super().__init__()
        self.mtrnn = MyMTRNNCell(in_dim, fast_dim, slow_dim, fast_tau, slow_tau)
        self.output = nn.Sequential(nn.Linear(fast_dim, out_dim), nn.Tanh())

    def forward(self, x, state=None):
        rnn_hid = self.mtrnn(x, state)
        y_hat = self.output(rnn_hid[0])
        return y_hat, rnn_hid

# 学習
my_mtrnn = MyBasicMTRNN(state_dim, 40, 10, 2, 20, state_dim)
my_opt = optim.Adam(my_mtrnn.parameters(), lr=0.001)
my_trainer = fullBPTTtrainer(my_mtrnn, my_opt)

my_losses = []
for epoch in range(200):
    loss = my_trainer.process_epoch(x_train, y_train)
    my_losses.append(loss)
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.6f}")

plt.semilogy(train_losses, label="LSTM", alpha=0.8)
plt.semilogy(my_losses, label="MTRNN", alpha=0.8)
plt.xlabel("エポック")
plt.ylabel("損失 (MSE, 対数)")
plt.title("LSTM vs MTRNN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

</details>

---
## まとめ

今回学んだ内容を振り返ります：

| トピック | ポイント |
|----------|----------|
| **RNN** | 隠れ状態を通じて時系列の情報を伝播する |
| **LSTM** | ゲート機構とセル状態で長期記憶を実現 |
| **BPTT** | 時間方向に計算グラフを展開して逆伝播 |
| **MTRNN** | 複数の時定数で階層的な時間表現を獲得 |

### 学習のポイント

1. **LSTMCell は1ステップずつ処理する**: 時刻ループの中で呼び出し、状態を引き継ぐ
2. **Full BPTT は系列全体で勾配を計算する**: メモリに注意が必要
3. **MTRNNは時間スケールの分離ができる**: Fast=瞬時の動き、Slow=動作の種類

### 次のステップ

- 画像入力（CNN + RNN）への拡張
- Attention機構の導入
- より複雑なロボット動作データへの適用